# Solve a small variational problem

Goal: minimize the energy of H=Z+0.1X with a one-qubit rotation ansatz. The classical optimizer proposes an angle. Each objective evaluation binds it at synthesis time, executes a FlagQuantum program and computes energy from the actual simulator state. This is a small, exact local simulation, not a hardware measurement or a runtime-parameterized training claim.

In [ ]:
import flagquantum as fq
from flagquantum import synth
import math
import torch
from flagquantum.algorithms import NelderMeadOptimizer
from flagquantum.synth import Output, QBit, R, allocate, qfunc

@qfunc
def ansatz(angle: float, q: Output[QBit]) -> None:
    allocate(q)
    R(angle, math.pi/2, q)

hamiltonian = torch.tensor([[1.0, 0.1], [0.1, -1.0]], dtype=torch.complex128)
evaluations = []

def energy(parameters: torch.Tensor) -> torch.Tensor:
    angle = float(parameters[0])
    program = synth.synthesize(ansatz, classical_inputs={"angle": angle}, max_qubits=1, max_workspace=0, memory_limit_bytes=1024**2)
    state = fq.run(program).statevector().flatten()
    value = torch.vdot(state, hamiltonian @ state).real
    evaluations.append((angle, float(value)))
    return value


## Optimize with actual executions

`R(angle, pi/2, q)` is the supported rotation about the Y axis. The first angle is zero; Nelder–Mead needs no gradients. The classical matrix contraction is explicit because this tiny tutorial reads an exact statevector.

In [ ]:
optimizer = NelderMeadOptimizer(maxiter=1000)
optimization = optimizer.minimize(energy, torch.zeros(1, dtype=torch.float64))
print("Converged:", optimization.converged)
print("Energy:", optimization.value)
print("Angle:", optimization.parameters.tolist())
print("Executed evaluations:", len(evaluations))


## Interpret and extend

The ground energy is -sqrt(1.01), about -1.00498756. Compare the final energy with the first recorded value to observe improvement. The optimizer receives only executed results; the known eigenvalue is used for independent acceptance, not substituted into the objective.

Try a different initial angle. Larger problems need appropriate observable measurement, state representations and parameter-binding support; this tutorial makes no scaling claim.